# Zero-DCE — Phase A: Noise-Propagation Diagnostics

Experiments 0–4 plus the Experiment 3 vs 4 consistency check. Diagnostics on the
**unmodified** pretrained baseline — no training, no loss changes, no new checkpoint.

**Branch:** `master` · **Checkpoint:** `Epoch199.pth` · **Data:** LOL eval15 (15 paired images)

### Recurrence under test
```
x_n = x_(n-1) + A_n * (x_(n-1)^2 - x_(n-1))
g_n = 1 + A_n * (2*x_(n-1) - 1)          local derivative
J_n = prod(g_i, i=1..n)                   cumulative, clean trajectory
```
The paper writes the curve as `I + aI(1-I)`; the code's `A` maps to `A = -a`.
All cells use the **code** convention.

### What was fixed relative to `ZERODCE_1.ipynb`
- `device`, `BRIGHTNESS_BINS`, `SIGMAS`, `N_TRIALS`, `BASE_SEED` and `LOW_DIR`-as-`Path`
  were never defined in any saved cell. Running top to bottom always raised
  `NameError: name 'device' is not defined` at the model-loading cell. They are now
  defined in the BOOTSTRAP cell.
- Repo setup, Drive mount, dataset download, model loading and the shared helpers
  are consolidated into that single BOOTSTRAP cell.
- Repeated `git pull` cells collapsed into one.
- `model_module` (the module) is now kept distinct from `model`/`net` (the network).
- Superseded diagnostics and the `W_noise` training sweeps were moved to
  `99_archive_superseded.ipynb`.

## BOOTSTRAP

Run first after every runtime restart. Must print `BOOTSTRAP OK`.

In [ ]:
# ============================================================
# BOOTSTRAP - run once per Colab runtime, before everything else
#
# Defines every global the Phase A cells rely on. In the original
# notebook some of these (device, BRIGHTNESS_BINS, SIGMAS, N_TRIALS,
# BASE_SEED, LOW_DIR-as-Path) were never saved in any cell, so the
# notebook could not be re-run from a cold start.
# ============================================================

import os, sys, json, hashlib, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from pathlib import Path
from PIL import Image
from datetime import datetime
from tqdm.auto import tqdm

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "pandas"], check=False)

# ------------------------------------------------------------
# 1. Repository - the only git pull in this notebook
# ------------------------------------------------------------
REPO_DIR = "/content/Zero-DCE"
BRANCH   = "master"
CODE_DIR = f"{REPO_DIR}/Zero-DCE_code"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/shreyaio/Zero-DCE.git", REPO_DIR],
        check=True,
    )

def git_run(args):
    return subprocess.check_output(["git"] + args, cwd=REPO_DIR, text=True).strip()

git_run(["fetch", "origin", BRANCH])
git_run(["checkout", BRANCH])
git_run(["pull", "--ff-only", "origin", BRANCH])

current_branch = git_run(["branch", "--show-current"])
commit = git_run(["rev-parse", "HEAD"])
print("Branch:", current_branch)
print("Commit:", commit)

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------
# 2. Device - BOTH spellings (the old cells disagreed, which is
#    exactly what broke the notebook)
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = device
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 3. Dataset - Path objects (experiment cells call .iterdir())
# ------------------------------------------------------------
LOL_ROOT = "/content/lol_dataset/lol_dataset/eval15"

if not os.path.isdir(f"{LOL_ROOT}/low"):
    from huggingface_hub import hf_hub_download
    import shutil
    os.makedirs("/content/lol_dataset", exist_ok=True)
    zip_path = hf_hub_download(repo_id="geekyrakshit/LoL-Dataset",
                               filename="lol_dataset.zip", repo_type="dataset")
    shutil.unpack_archive(zip_path, "/content/lol_dataset")

LOW_DIR  = Path(f"{LOL_ROOT}/low")
HIGH_DIR = Path(f"{LOL_ROOT}/high")
assert LOW_DIR.is_dir() and HIGH_DIR.is_dir(), "LOL eval15 not found"

low_files  = sorted(f for f in os.listdir(LOW_DIR)  if not f.startswith("."))
high_files = sorted(f for f in os.listdir(HIGH_DIR) if not f.startswith("."))
assert low_files == high_files, "LOW/HIGH filenames do not match"
print("Paired images:", len(low_files))

# NOTE: only LOW images are fed to the network.
#       HIGH images are paired references only.

# ------------------------------------------------------------
# 4. Experiment configuration
#
# RECONSTRUCTED - these were never saved in the original notebook.
# BASE_SEED in particular is a best guess: it changes which noise
# draws occur, so Experiment 2/3 reproduce statistically but not
# bit-for-bit. If an old experiment_*_metadata.json on Drive records
# the real seed, substitute it here.
# ------------------------------------------------------------
BRIGHTNESS_BINS = {
    "dark":       (0.00, 0.10),
    "mid_dark":   (0.10, 0.25),
    "mid_bright": (0.25, 0.50),
    "bright":     (0.50, 1.01),
}

SIGMAS    = [0.005, 0.010, 0.020]
N_TRIALS  = 30
BASE_SEED = 42026

# Output roots. RESULTS_DIR was another name the original notebook used
# but never defined; Experiment 2 writes its experiment_2_corrected_*
# files here.
PHASE_A_DIR = Path("/content/drive/MyDrive/zero-dce/results/phase_a")
RESULTS_DIR = PHASE_A_DIR / "experiment_2"
PHASE_A_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Phase A results root:", PHASE_A_DIR)

# ------------------------------------------------------------
# 5. Model
#    model_module = the module,  model / net = the loaded network
# ------------------------------------------------------------
import model as model_module

CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
assert os.path.isfile(CKPT), f"Checkpoint missing: {CKPT}"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

model = model_module.enhance_net_nopool().to(device)
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()
net = model

params = sum(p.numel() for p in model.parameters())
ckpt_hash = sha256_file(CKPT)
print("Parameters:", params)
print("Checkpoint SHA256:", ckpt_hash)

assert params == 79416, f"Expected 79416 parameters, got {params}"
EXPECTED_HASH = "7c211c511a5213a898716cf65bee964e62ff04ffeb2991d20d7b5a28f7692154"
if ckpt_hash != EXPECTED_HASH:
    print("WARNING: checkpoint hash differs from the Phase A record.")

# ------------------------------------------------------------
# 6. Shared helper functions (was cell 37)
# ------------------------------------------------------------
def load_rgb_image(path):
    """Load an image as an RGB float tensor in [0, 1]."""
    image = Image.open(path).convert("RGB")
    array = np.asarray(image).astype(np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)
    return tensor.to(device)


@torch.no_grad()
def predict_clean_curves(clean):
    """Predict curves from the clean image."""
    _, enhanced, curves = model(clean)
    return enhanced, curves


@torch.no_grad()
def apply_zero_dce_curves(image, curves):
    """Apply already-predicted curves, held fixed across 8 iterations."""
    batch, _, height, width = image.shape
    curve_steps = curves.reshape(batch, 8, 3, height, width)
    x = image.clone()
    for i in range(8):
        A = curve_steps[:, i]
        x = x + A * (x * x - x)
    return x


@torch.no_grad()
def calculate_fixed_curve_jacobian(clean, curves):
    """J = prod_n [1 + A_n (2 x_(n-1) - 1)] along the clean trajectory."""
    batch, _, height, width = clean.shape
    curve_steps = curves.reshape(batch, 8, 3, height, width)
    x = clean.clone()
    J = torch.ones_like(clean)
    for i in range(8):
        A = curve_steps[:, i]
        J = J * (1.0 + A * (2.0 * x - 1.0))
        x = x + A * (x * x - x)
    return J


def variance_from_sums(sum_x, sum_x2, n):
    """Unbiased sample variance, pixel by pixel."""
    return torch.clamp((sum_x2 - (sum_x * sum_x) / n) / (n - 1), min=0.0)


def safe_mean(tensor):
    return float(tensor.mean().item())


def brightness_region_metrics(variance_map, clean):
    """Average across RGB, then aggregate by clean-image luminance."""
    clean_luminance = (0.299 * clean[:, 0:1]
                       + 0.587 * clean[:, 1:2]
                       + 0.114 * clean[:, 2:3])
    variance_luminance = variance_map.mean(dim=1, keepdim=True)

    results = {}
    for name, (lower, upper) in BRIGHTNESS_BINS.items():
        mask = (clean_luminance >= lower) & (clean_luminance < upper)
        results[name] = safe_mean(variance_luminance[mask]) if mask.any() else np.nan
    return results


# ------------------------------------------------------------
# 7. Smoke test - manual recurrence must match the model output
# ------------------------------------------------------------
_probe = load_rgb_image(sorted(LOW_DIR.glob("*.png"))[0])
with torch.no_grad():
    _, _enh, _curves = model(_probe)
_err = (apply_zero_dce_curves(_probe, _curves) - _enh).abs().max().item()

print(f"\nRecurrence smoke test - max abs error: {_err:.3e}")
print("BOOTSTRAP OK" if _err < 1e-5 else "BOOTSTRAP FAILED - stop here")

## Experiment 0 — Baseline verification

Confirms the checkpoint, the curve tensor shape, and that the manual eight-step
recurrence reproduces the model's own output exactly.

*Expected: reconstruction MAE 0, max error 0, mean gain per iteration
1.3157 → 1.1626, curve range −0.474989 … −0.071696.*

In [ ]:
# ============================================================
# EXPERIMENT 0 - BASELINE VERIFICATION
#
# Verifies the checkpoint, the model outputs, the curve tensor
# and the eight-step recurrence.
#
# Repo/Drive/dataset/model setup now lives in the BOOTSTRAP cell.
# ============================================================

import inspect

# -----------------------------
# 3. Inspect actual forward()
# -----------------------------
print("\nACTUAL MODEL FORWARD IMPLEMENTATION")
print("-----------------------------------")
print(inspect.getsource(model_module.enhance_net_nopool.forward))


# -----------------------------
# 6. Helper functions
# -----------------------------
def rgb_luminance(img):
    """Input: H x W x 3 RGB image."""
    return (
        0.2126 * img[..., 0] +
        0.7152 * img[..., 1] +
        0.0722 * img[..., 2]
    )

def luminance_stats(img):
    y = rgb_luminance(img)
    return {
        "mean": float(y.mean()),
        "std": float(y.std()),
        "p05": float(np.percentile(y, 5)),
        "p50": float(np.percentile(y, 50)),
        "p95": float(np.percentile(y, 95)),
        "min": float(y.min()),
        "max": float(y.max())
    }

def gain_stats(g):
    g = g.detach().float().reshape(-1)

    return {
        "mean": float(g.mean().item()),
        "median": float(g.median().item()),
        "p05": float(torch.quantile(g, 0.05).item()),
        "p95": float(torch.quantile(g, 0.95).item()),
        "fraction_gt_1": float((g > 1).float().mean().item()),
        "fraction_lt_1": float((g < 1).float().mean().item()),
        "fraction_negative": float((g < 0).float().mean().item())
    }


# -----------------------------
# 7. Verify recurrence and gains
# -----------------------------
image_records = []
iteration_records = [[] for _ in range(8)]

for idx, fname in enumerate(low_files):

    # LOW-light input only
    img = Image.open(
        os.path.join(LOW_DIR, fname)
    ).convert("RGB")

    x_np = np.asarray(img).astype(np.float32) / 255.0

    x0 = torch.from_numpy(x_np).permute(
        2, 0, 1
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = net(x0)

    # Verify against the existing notebook's forward convention.
    assert isinstance(outputs, (tuple, list)), (
        f"Unexpected model output type: {type(outputs)}"
    )
    assert len(outputs) == 3, (
        f"Expected 3 outputs, got {len(outputs)}"
    )

    enhanced = outputs[1]
    r = outputs[2]

    B, C, H, W = x0.shape

    assert enhanced.shape == x0.shape, (
        f"Enhanced shape mismatch: {enhanced.shape}"
    )
    assert r.shape == (1, 24, H, W), (
        f"Unexpected curve shape: {r.shape}"
    )

    # Eight iterations, three RGB channels each.
    r_iters = r.reshape(1, 8, 3, H, W)

    # Reconstruct the enhancement from the actual curve maps.
    x_prev = x0.clone()

    reconstructed_stages = [x_prev.clone()]

    for n in range(8):

        A = r_iters[:, n]

        # Local derivative, holding A fixed.
        g = 1.0 + A * (2.0 * x_prev - 1.0)

        stats = gain_stats(g)
        stats["iteration"] = n + 1
        stats["image"] = fname

        iteration_records[n].append(stats)

        # Actual repository recurrence:
        # x_n = x_(n-1) + A_n * (x_(n-1)^2 - x_(n-1))
        x_prev = x_prev + A * (x_prev.square() - x_prev)

        reconstructed_stages.append(x_prev.clone())

    # Compare reconstructed output against model output.
    reconstruction_error = (
        reconstructed_stages[-1] - enhanced
    ).abs()

    rec_max = float(reconstruction_error.max().item())
    rec_mean = float(reconstruction_error.mean().item())

    # Convert images to HWC for luminance analysis.
    input_np = (
        x0.squeeze(0).permute(1, 2, 0).cpu().numpy()
    )

    enhanced_np = (
        enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy()
    )

    reconstructed_np = (
        reconstructed_stages[-1]
        .squeeze(0)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )

    curve_np = r.detach().cpu().numpy()

    image_records.append({
        "filename": fname,
        "height": H,
        "width": W,
        "curve_min": float(curve_np.min()),
        "curve_max": float(curve_np.max()),
        "curve_mean": float(curve_np.mean()),
        "curve_abs_mean": float(np.abs(curve_np).mean()),
        "reconstruction_max_abs_error": rec_max,
        "reconstruction_mean_abs_error": rec_mean,
        "input_luminance": luminance_stats(input_np),
        "enhanced_luminance": luminance_stats(enhanced_np),
        "reconstructed_luminance": luminance_stats(
            reconstructed_np
        ),
        "enhanced_fraction_below_0": float(
            (enhanced_np < 0).mean()
        ),
        "enhanced_fraction_above_1": float(
            (enhanced_np > 1).mean()
        )
    })

    if (idx + 1) % 5 == 0:
        print(f"Processed {idx + 1}/{len(low_files)} images")


# -----------------------------
# 8. Aggregate per-iteration data
# -----------------------------
aggregate_gains = []

for n, records in enumerate(iteration_records):

    aggregate = {"iteration": n + 1}

    for key in [
        "mean",
        "median",
        "p05",
        "p95",
        "fraction_gt_1",
        "fraction_lt_1",
        "fraction_negative"
    ]:
        aggregate[key] = float(
            np.mean([record[key] for record in records])
        )

    aggregate_gains.append(aggregate)


# -----------------------------
# 9. Print results
# -----------------------------
print("\n" + "=" * 90)
print("EXPERIMENT 0 — BASELINE VERIFICATION RESULTS")
print("=" * 90)

print("\nPER-ITERATION LOCAL DERIVATIVE GAIN")
print(
    f"{'Iter':>5} {'Mean':>9} {'Median':>9} "
    f"{'P05':>9} {'P95':>9} "
    f"{'Gain>1':>10} {'Gain<1':>10} {'Negative':>10}"
)

for a in aggregate_gains:
    print(
        f"{a['iteration']:5d} "
        f"{a['mean']:9.4f} "
        f"{a['median']:9.4f} "
        f"{a['p05']:9.4f} "
        f"{a['p95']:9.4f} "
        f"{a['fraction_gt_1']*100:9.2f}% "
        f"{a['fraction_lt_1']*100:9.2f}% "
        f"{a['fraction_negative']*100:9.2f}%"
    )

print("\nRECONSTRUCTION ACCURACY")
print(
    "Mean reconstruction MAE:",
    np.mean([
        x["reconstruction_mean_abs_error"]
        for x in image_records
    ])
)
print(
    "Maximum reconstruction error:",
    max([
        x["reconstruction_max_abs_error"]
        for x in image_records
    ])
)

print("\nLUMINANCE STATISTICS")
for key in ["input_luminance", "enhanced_luminance"]:
    means = [
        x[key]["mean"] for x in image_records
    ]
    stds = [
        x[key]["std"] for x in image_records
    ]
    print(
        key,
        "| mean luminance:", round(float(np.mean(means)), 5),
        "| mean std:", round(float(np.mean(stds)), 5)
    )

print("\nCURVE PARAMETER RANGE")
print(
    "Global min:",
    min(x["curve_min"] for x in image_records)
)
print(
    "Global max:",
    max(x["curve_max"] for x in image_records)
)


# -----------------------------
# 10. Save complete results
# -----------------------------
OUT_DIR = (
    "/content/drive/MyDrive/zero-dce/"
    "results/phase_a/experiment_0"
)
os.makedirs(OUT_DIR, exist_ok=True)

results = {
    "experiment": "Phase A - Experiment 0",
    "repository_branch": current_branch,
    "repository_commit": commit,
    "checkpoint": CKPT,
    "checkpoint_sha256": sha256_file(CKPT),
    "parameter_count": params,
    "dataset": "LOL eval15",
    "input_directory": str(LOW_DIR),
    "reference_directory": str(HIGH_DIR),
    "number_of_images": len(low_files),
    "recurrence": "x_n = x_(n-1) + A_n*(x_(n-1)^2 - x_(n-1))",
    "local_gain": "g_n = 1 + A_n*(2*x_(n-1) - 1)",
    "iteration_gain_summary": aggregate_gains,
    "per_image": image_records,
    "per_image_iteration_gains": iteration_records
}

RESULT_PATH = f"{OUT_DIR}/baseline_verification.json"

with open(RESULT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 90)
print("EXPERIMENT 0 COMPLETED")
print("Saved:", RESULT_PATH)
print("=" * 90)

## Experiment 1 — Controlled scalar curve response

Isolates the recurrence from the network: pure Monte-Carlo on scalars.

*Expected: amplification 19.39x at signal 0.05 falling to 0.133x at 0.75 —
brightness dependence, with the first-order prediction accurate to <1%.*

In [ ]:

# ============================================================
# EXPERIMENT 1 — CONTROLLED GAIN-RESPONSE SWEEP
# Zero-DCE | Phase A
#
# Purpose:
# 1. Compare empirical noise variance with derivative prediction.
# 2. Measure nonlinear noise-induced brightness bias.
# 3. Measure local contrast preservation.
#
# No training. No model modifications.
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# -----------------------------
# 1. Experimental configuration
# -----------------------------

SEED = 42
N_SAMPLES = 100_000
N_ITERATIONS = 8

A_VALUES = [-0.10, -0.25, -0.40]

SIGNAL_LEVELS = [
    0.05, 0.10, 0.25, 0.50, 0.75
]

NOISE_LEVELS = [
    0.002, 0.005, 0.010
]

DETAIL_DELTA = 0.005

rng = np.random.default_rng(SEED)

# -----------------------------
# 2. Actual Zero-DCE recurrence
# -----------------------------

def zerodce_curve(x, A, iterations=8):
    """
    Applies the actual repository recurrence:

    x_n = x_(n-1) + A*(x_(n-1)^2 - x_(n-1))

    A is held constant across iterations.
    """

    x = np.asarray(x, dtype=np.float64).copy()

    stages = [x.copy()]

    for _ in range(iterations):
        x = x + A * (x**2 - x)
        stages.append(x.copy())

    return x, stages


# -----------------------------
# 3. First-order derivative
# -----------------------------

def derivative_prediction(signal, A, iterations=8):
    """
    Calculates the cumulative local Jacobian:

    J = product of g_n

    g_n = 1 + A*(2*x_(n-1)-1)

    Predicted variance = sigma^2 * J^2
    """

    x = np.asarray(signal, dtype=np.float64).copy()

    J = np.ones_like(x)

    for _ in range(iterations):

        g = 1.0 + A * (2.0*x - 1.0)

        J *= g

        x = x + A * (x**2 - x)

    return x, J


# -----------------------------
# 4. Run controlled sweep
# -----------------------------

results = []

print("Starting Experiment 1...")
print("Monte Carlo samples:", N_SAMPLES)
print("Iterations:", N_ITERATIONS)
print("Random seed:", SEED)

for signal in SIGNAL_LEVELS:

    for sigma in NOISE_LEVELS:

        # Identical standard Gaussian realization
        # for all A conditions at this signal/noise level.
        z = rng.standard_normal(N_SAMPLES)

        epsilon = sigma * z

        noisy_input = signal + epsilon

        for A in A_VALUES:

            # --------------------------------
            # Clean signal transformation
            # --------------------------------

            clean_output, clean_stages = zerodce_curve(
                signal, A, N_ITERATIONS
            )

            # --------------------------------
            # Noisy signal transformation
            # --------------------------------

            noisy_output, noisy_stages = zerodce_curve(
                noisy_input, A, N_ITERATIONS
            )

            # --------------------------------
            # Empirical noise response
            # --------------------------------

            residual = noisy_output - clean_output

            empirical_variance = float(
                np.var(residual)
            )

            empirical_std = float(
                np.std(residual)
            )

            # --------------------------------
            # Derivative prediction
            # --------------------------------

            _, J = derivative_prediction(
                signal, A, N_ITERATIONS
            )

            predicted_variance = float(
                sigma**2 * J**2
            )

            predicted_amplification = float(
                J**2
            )

            empirical_amplification = (
                empirical_variance / sigma**2
            )

            # --------------------------------
            # Nonlinear brightness bias
            # --------------------------------

            mean_shift = float(
                np.mean(noisy_output) - clean_output
            )

            # --------------------------------
            # Input clipping / range diagnostic
            # No clipping is applied.
            # --------------------------------

            input_out_of_range = float(
                np.mean(
                    (noisy_input < 0) |
                    (noisy_input > 1)
                )
            )

            output_out_of_range = float(
                np.mean(
                    (noisy_output < 0) |
                    (noisy_output > 1)
                )
            )

            # --------------------------------
            # Deterministic contrast preservation
            # --------------------------------

            lower = signal - DETAIL_DELTA
            upper = signal + DETAIL_DELTA

            lower_out, _ = zerodce_curve(
                lower, A, N_ITERATIONS
            )

            upper_out, _ = zerodce_curve(
                upper, A, N_ITERATIONS
            )

            input_contrast = upper - lower

            output_contrast = upper_out - lower_out

            contrast_retention = float(
                output_contrast / input_contrast
            )

            # --------------------------------
            # Store all observations
            # --------------------------------

            results.append({

                "signal": signal,
                "sigma": sigma,
                "A": A,

                "input_variance": sigma**2,

                "empirical_output_variance":
                    empirical_variance,

                "predicted_output_variance":
                    predicted_variance,

                "empirical_amplification":
                    float(empirical_amplification),

                "predicted_amplification":
                    predicted_amplification,

                "cumulative_jacobian":
                    float(J),

                "empirical_output_std":
                    empirical_std,

                "mean_brightness_shift":
                    mean_shift,

                "clean_output":
                    float(clean_output),

                "contrast_retention":
                    contrast_retention,

                "input_out_of_range_fraction":
                    input_out_of_range,

                "output_out_of_range_fraction":
                    output_out_of_range
            })


# -----------------------------
# 5. Create results dataframe
# -----------------------------

df = pd.DataFrame(results)

df["variance_prediction_error_pct"] = (
    100.0 *
    (
        df["empirical_output_variance"] -
        df["predicted_output_variance"]
    ) /
    df["predicted_output_variance"]
)

print("\n" + "="*100)
print("EXPERIMENT 1 — COMPLETE RESULTS")
print("="*100)

display(
    df.round(7)
)

# -----------------------------
# 6. Aggregate by curve parameter
# -----------------------------

print("\nAGGREGATE RESULTS BY CURVE PARAMETER")

summary = df.groupby("A").agg({

    "empirical_amplification": "mean",

    "predicted_amplification": "mean",

    "variance_prediction_error_pct": "mean",

    "mean_brightness_shift": "mean",

    "contrast_retention": "mean"

}).reset_index()

display(summary.round(6))

# -----------------------------
# 7. Aggregate by signal intensity
# -----------------------------

print("\nAGGREGATE RESULTS BY SIGNAL INTENSITY")

signal_summary = df.groupby("signal").agg({

    "empirical_amplification": "mean",

    "predicted_amplification": "mean",

    "mean_brightness_shift": "mean",

    "contrast_retention": "mean"

}).reset_index()

display(signal_summary.round(6))

# -----------------------------
# 8. Save results to Drive
# -----------------------------

OUT_DIR = (
    "/content/drive/MyDrive/zero-dce/"
    "results/phase_a/experiment_1"
)

os.makedirs(OUT_DIR, exist_ok=True)

csv_path = os.path.join(
    OUT_DIR,
    "controlled_gain_response.csv"
)

json_path = os.path.join(
    OUT_DIR,
    "controlled_gain_response.json"
)

df.to_csv(csv_path, index=False)

metadata = {

    "experiment": "Phase A - Experiment 1",

    "seed": SEED,

    "monte_carlo_samples": N_SAMPLES,

    "iterations": N_ITERATIONS,

    "curve_parameters": A_VALUES,

    "signal_levels": SIGNAL_LEVELS,

    "noise_levels": NOISE_LEVELS,

    "noise_model": "Additive zero-mean Gaussian",

    "input_clipping": False,

    "recurrence":
        "x_n = x_(n-1) + A*(x_(n-1)^2-x_(n-1))",

    "derivative":
        "g_n = 1 + A*(2*x_(n-1)-1)",

    "prediction":
        "Var_out approximately sigma^2 * product(g_n^2)",

    "results": results
}

with open(json_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("\nEXPERIMENT 1 FINISHED")
print("CSV:", csv_path)
print("JSON:", json_path)


## Experiment 2 — Frozen curves (2A) vs end-to-end (2B)

Separates the curve transform itself from the network's curve re-prediction.
Measures **pixelwise temporal variance** across trials.

*Expected: 78.35x / 82.65x / 104.22x frozen at sigma = 0.005 / 0.010 / 0.020.*

In [ ]:

# ============================================================
# EXPERIMENT 2 — CORRECTED
# Frozen curves (2A) vs End-to-end inference (2B)
# ============================================================

image_paths = sorted([
    p for p in LOW_DIR.iterdir()
    if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]
])

all_rows = []
curve_rows = []

metadata = {
    "experiment": "Experiment 2 - Corrected",
    "date": datetime.now().isoformat(),
    "trials_per_image_per_sigma": N_TRIALS,
    "sigmas": SIGMAS,
    "base_seed": BASE_SEED,
    "input_noise_clipping": False,
    "output_clipping": "Measured separately; raw outputs retained",
    "brightness_bins": BRIGHTNESS_BINS,
    "images": []
}

# ------------------------------------------------------------
# PROCESS EACH IMAGE
# ------------------------------------------------------------

for image_index, image_path in enumerate(
    tqdm(image_paths, desc="Images")
):

    image_name = image_path.name

    # Load clean image
    clean = load_rgb_image(image_path)

    # Predict curves once from clean image
    with torch.no_grad():
        _, clean_curves = predict_clean_curves(clean)

        # Clean baseline output
        clean_output = apply_zero_dce_curves(
            clean, clean_curves
        )

        # Verify manual recurrence against original model
        _, model_clean_output, _ = model(clean)

    recurrence_error = (
        clean_output - model_clean_output
    ).abs().max().item()

    if recurrence_error > 1e-5:
        raise RuntimeError(
            f"Recurrence mismatch for {image_name}: "
            f"{recurrence_error}"
        )

    # Calculate fixed-curve local Jacobian
    with torch.no_grad():
        J = calculate_fixed_curve_jacobian(
            clean, clean_curves
        )

    # Clean output clipped baseline
    clean_output_clipped = torch.clamp(
        clean_output, 0.0, 1.0
    )

    # Move clean reference tensors to CPU
    clean_cpu = clean.detach().cpu().double()
    J_cpu = J.detach().cpu().double()

    height, width = clean.shape[-2:]

    metadata["images"].append({
        "image": image_name,
        "height": height,
        "width": width,
        "recurrence_max_error": recurrence_error
    })

    # --------------------------------------------------------
    # PROCESS EACH NOISE LEVEL
    # --------------------------------------------------------

    for sigma_index, sigma in enumerate(SIGMAS):

        # Online accumulation of pixel-wise measurements.
        # We do not retain all 30 output images in memory.

        sums = {
            "input_noise": torch.zeros_like(clean_cpu),

            "frozen_residual": torch.zeros_like(clean_cpu),
            "end_to_end_residual": torch.zeros_like(clean_cpu),

            "frozen_clipped_residual": torch.zeros_like(clean_cpu),
            "end_to_end_clipped_residual": torch.zeros_like(clean_cpu)
        }

        sums_sq = {
            key: torch.zeros_like(value)
            for key, value in sums.items()
        }

        # Curve changes across the eight iterations
        curve_abs_delta_sum = torch.zeros(
            8, dtype=torch.float64
        )

        curve_sq_delta_sum = torch.zeros(
            8, dtype=torch.float64
        )

        # Per-trial statistics
        frozen_mean_residuals = []
        end_to_end_mean_residuals = []

        frozen_oor_fractions = []
        end_to_end_oor_fractions = []

        # ----------------------------------------------------
        # 30 INDEPENDENT NOISE REALIZATIONS
        # ----------------------------------------------------

        for trial in range(N_TRIALS):

            # Unique reproducible seed for every image,
            # noise level and trial.
            seed = (
                BASE_SEED
                + image_index * 10000000
                + sigma_index * 100000
                + trial
            )

            generator = torch.Generator(device=device)
            generator.manual_seed(seed)

            # Generate Gaussian noise
            noise = torch.randn(
                clean.shape,
                generator=generator,
                device=device,
                dtype=clean.dtype
            ) * sigma

            # Do not clip noisy input
            noisy = clean + noise

            # ------------------------------------------------
            # 2A: FROZEN CURVES
            # Apply clean-image curves to noisy input
            # ------------------------------------------------

            with torch.no_grad():
                frozen_output = apply_zero_dce_curves(
                    noisy, clean_curves
                )

            # ------------------------------------------------
            # 2B: END-TO-END INFERENCE
            # Predict new curves from noisy input
            # ------------------------------------------------

            with torch.no_grad():
                _, end_to_end_output, noisy_curves = model(
                    noisy
                )

            # ------------------------------------------------
            # CURVE MAP SENSITIVITY
            # ------------------------------------------------

            clean_curve_steps = clean_curves.reshape(
                1, 8, 3, height, width
            )

            noisy_curve_steps = noisy_curves.reshape(
                1, 8, 3, height, width
            )

            curve_delta = (
                noisy_curve_steps - clean_curve_steps
            )

            curve_abs_delta_sum += (
                curve_delta.abs()
                .mean(dim=(0, 2, 3, 4))
                .detach()
                .cpu()
                .double()
            )

            curve_sq_delta_sum += (
                curve_delta.square()
                .mean(dim=(0, 2, 3, 4))
                .detach()
                .cpu()
                .double()
            )

            # ------------------------------------------------
            # RAW OUTPUT RESIDUALS
            # ------------------------------------------------

            frozen_residual = (
                frozen_output - clean_output
            )

            end_to_end_residual = (
                end_to_end_output - clean_output
            )

            # ------------------------------------------------
            # CLIPPED OUTPUT RESIDUALS
            # Subtract the corresponding clean clipped output
            # ------------------------------------------------

            frozen_clipped = torch.clamp(
                frozen_output, 0.0, 1.0
            )

            end_to_end_clipped = torch.clamp(
                end_to_end_output, 0.0, 1.0
            )

            frozen_clipped_residual = (
                frozen_clipped - clean_output_clipped
            )

            end_to_end_clipped_residual = (
                end_to_end_clipped - clean_output_clipped
            )

            # ------------------------------------------------
            # ACCUMULATE PIXEL-WISE TEMPORAL STATISTICS
            # ------------------------------------------------

            trial_data = {
                "input_noise": noise,

                "frozen_residual": frozen_residual,
                "end_to_end_residual": end_to_end_residual,

                "frozen_clipped_residual": frozen_clipped_residual,
                "end_to_end_clipped_residual": end_to_end_clipped_residual
            }

            for key, value in trial_data.items():

                value_cpu = (
                    value.detach()
                    .cpu()
                    .double()
                )

                sums[key] += value_cpu
                sums_sq[key] += value_cpu.square()

            # Mean residual for this trial
            frozen_mean_residuals.append(
                float(frozen_residual.mean().item())
            )

            end_to_end_mean_residuals.append(
                float(end_to_end_residual.mean().item())
            )

            # Fraction of output values outside [0, 1]
            frozen_oor_fractions.append(
                float(
                    (
                        (frozen_output < 0)
                        | (frozen_output > 1)
                    ).float().mean().item()
                )
            )

            end_to_end_oor_fractions.append(
                float(
                    (
                        (end_to_end_output < 0)
                        | (end_to_end_output > 1)
                    ).float().mean().item()
                )
            )

            # Release trial-specific tensors
            del (
                noise,
                noisy,
                frozen_output,
                end_to_end_output,
                noisy_curves,
                frozen_residual,
                end_to_end_residual
            )

        # ----------------------------------------------------
        # CALCULATE PIXEL-WISE TEMPORAL VARIANCE
        # ----------------------------------------------------

        variances = {
            key: variance_from_sums(
                sums[key],
                sums_sq[key],
                N_TRIALS
            )
            for key in sums
        }

        # Average variance over RGB channels and pixels
        input_variance = safe_mean(
            variances["input_noise"]
        )

        frozen_variance = safe_mean(
            variances["frozen_residual"]
        )

        end_to_end_variance = safe_mean(
            variances["end_to_end_residual"]
        )

        frozen_clipped_variance = safe_mean(
            variances["frozen_clipped_residual"]
        )

        end_to_end_clipped_variance = safe_mean(
            variances["end_to_end_clipped_residual"]
        )

        # ----------------------------------------------------
        # FIRST-ORDER JACOBIAN PREDICTION
        # ----------------------------------------------------

        predicted_variance_map = (
            variances["input_noise"] * J_cpu.square()
        )

        predicted_variance = safe_mean(
            predicted_variance_map
        )

        # ----------------------------------------------------
        # REGIONAL VARIANCE ANALYSIS
        # Regions are defined using clean-image luminance
        # ----------------------------------------------------

        frozen_regions = brightness_region_metrics(
            variances["frozen_residual"],
            clean_cpu
        )

        end_to_end_regions = brightness_region_metrics(
            variances["end_to_end_residual"],
            clean_cpu
        )

        input_regions = brightness_region_metrics(
            variances["input_noise"],
            clean_cpu
        )

        predicted_regions = brightness_region_metrics(
            predicted_variance_map,
            clean_cpu
        )

        # ----------------------------------------------------
        # BUILD RESULT ROW
        # ----------------------------------------------------

        row = {
            "image": image_name,
            "sigma": sigma,
            "trials": N_TRIALS,

            "input_temporal_variance": input_variance,

            "frozen_output_temporal_variance": frozen_variance,

            "end_to_end_output_temporal_variance": end_to_end_variance,

            "frozen_variance_amplification": (
                frozen_variance / input_variance
                if input_variance > 0 else np.nan
            ),

            "end_to_end_variance_amplification": (
                end_to_end_variance / input_variance
                if input_variance > 0 else np.nan
            ),

            "first_order_predicted_variance": predicted_variance,

            "first_order_predicted_amplification": (
                predicted_variance / input_variance
                if input_variance > 0 else np.nan
            ),

            "frozen_first_order_relative_error_percent": (
                abs(frozen_variance - predicted_variance)
                / max(predicted_variance, 1e-15)
                * 100
            ),

            "frozen_clipped_temporal_variance": (
                frozen_clipped_variance
            ),

            "end_to_end_clipped_temporal_variance": (
                end_to_end_clipped_variance
            ),

            "frozen_mean_residual": np.mean(
                frozen_mean_residuals
            ),

            "end_to_end_mean_residual": np.mean(
                end_to_end_mean_residuals
            ),

            "frozen_output_out_of_range_fraction": np.mean(
                frozen_oor_fractions
            ),

            "end_to_end_output_out_of_range_fraction": np.mean(
                end_to_end_oor_fractions
            ),

            "curve_mean_abs_delta": float(
                (
                    curve_abs_delta_sum / N_TRIALS
                ).mean().item()
            ),

            "curve_mean_rms_delta": float(
                torch.sqrt(
                    curve_sq_delta_sum / N_TRIALS
                ).mean().item()
            )
        }

        # Add regional values to the result row
        for region in BRIGHTNESS_BINS:

            row[f"input_var_{region}"] = (
                input_regions[region]
            )

            row[f"frozen_var_{region}"] = (
                frozen_regions[region]
            )

            row[f"end_to_end_var_{region}"] = (
                end_to_end_regions[region]
            )

            row[f"predicted_var_{region}"] = (
                predicted_regions[region]
            )

        all_rows.append(row)

        # ----------------------------------------------------
        # SAVE PER-ITERATION CURVE CHANGES
        # ----------------------------------------------------

        for iteration in range(8):

            curve_rows.append({
                "image": image_name,
                "sigma": sigma,
                "iteration": iteration + 1,

                "mean_absolute_curve_delta": float(
                    (
                        curve_abs_delta_sum[iteration]
                        / N_TRIALS
                    ).item()
                ),

                "rms_curve_delta": float(
                    torch.sqrt(
                        curve_sq_delta_sum[iteration]
                        / N_TRIALS
                    ).item()
                )
            })

        # ----------------------------------------------------
        # PROGRESS OUTPUT
        # ----------------------------------------------------

        print(
            f"{image_name} | sigma={sigma:.3f} | "
            f"2A={row['frozen_variance_amplification']:.3f}x | "
            f"2B={row['end_to_end_variance_amplification']:.3f}x | "
            f"Jacobian={row['first_order_predicted_amplification']:.3f}x"
        )

        # Release sigma-specific tensors
        del sums, sums_sq, variances
        del predicted_variance_map
        del frozen_regions, end_to_end_regions
        del input_regions, predicted_regions

    # --------------------------------------------------------
    # RELEASE IMAGE-SPECIFIC GPU MEMORY
    # --------------------------------------------------------

    del clean
    del clean_curves
    del clean_output
    del model_clean_output
    del J
    del clean_output_clipped

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\nCorrected Experiment 2 completed successfully.")


In [ ]:
# Convert collected results to DataFrames
results_df = pd.DataFrame(all_rows)
curves_df = pd.DataFrame(curve_rows)

# Save detailed per-image results
per_image_path = RESULTS_DIR / "experiment_2_corrected_per_image.csv"
curve_path = RESULTS_DIR / "experiment_2_corrected_curve_changes.csv"

results_df.to_csv(per_image_path, index=False)
curves_df.to_csv(curve_path, index=False)

# Aggregate across images.
# Each image contributes equally to the mean.
summary_df = (
    results_df
    .groupby("sigma")
    .agg({
        "input_temporal_variance": "mean",
        "frozen_output_temporal_variance": "mean",
        "end_to_end_output_temporal_variance": "mean",
        "frozen_variance_amplification": "mean",
        "end_to_end_variance_amplification": "mean",
        "first_order_predicted_amplification": "mean",
        "frozen_first_order_relative_error_percent": "mean",
        "frozen_clipped_temporal_variance": "mean",
        "end_to_end_clipped_temporal_variance": "mean",
        "frozen_mean_residual": "mean",
        "end_to_end_mean_residual": "mean",
        "frozen_output_out_of_range_fraction": "mean",
        "end_to_end_output_out_of_range_fraction": "mean",
        "curve_mean_abs_delta": "mean",
        "curve_mean_rms_delta": "mean",
    })
    .reset_index()
)

summary_path = RESULTS_DIR / "experiment_2_corrected_summary.csv"
summary_df.to_csv(summary_path, index=False)

# Aggregate regional temporal variance
region_columns = [
    col for col in results_df.columns
    if col.startswith((
        "input_var_",
        "frozen_var_",
        "end_to_end_var_",
        "predicted_var_"
    ))
]

regional_summary = (
    results_df
    .groupby("sigma")[region_columns]
    .mean()
    .reset_index()
)

regional_path = RESULTS_DIR / "experiment_2_corrected_regional_summary.csv"
regional_summary.to_csv(regional_path, index=False)

# Aggregate curve changes by iteration and sigma
curve_summary = (
    curves_df
    .groupby(["sigma", "iteration"])
    .agg({
        "mean_absolute_curve_delta": "mean",
        "rms_curve_delta": "mean",
    })
    .reset_index()
)

curve_summary_path = RESULTS_DIR / "experiment_2_corrected_curve_summary.csv"
curve_summary.to_csv(curve_summary_path, index=False)

# Save metadata
metadata["number_of_images"] = len(image_paths)
metadata["number_of_trials_per_sigma"] = N_TRIALS
metadata["total_image_sigma_trial_combinations"] = (
    len(image_paths) * len(SIGMAS) * N_TRIALS
)

metadata_path = RESULTS_DIR / "experiment_2_corrected_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("All results saved.")

for path in [
    per_image_path,
    curve_path,
    summary_path,
    regional_path,
    curve_summary_path,
    metadata_path,
]:
    print(path)

In [ ]:
display_columns = [
    "sigma",
    "frozen_variance_amplification",
    "end_to_end_variance_amplification",
    "first_order_predicted_amplification",
    "frozen_first_order_relative_error_percent",
    "curve_mean_abs_delta",
    "frozen_output_out_of_range_fraction",
    "end_to_end_output_out_of_range_fraction",
]

display(
    summary_df[display_columns].round(6)
)

In [ ]:
display(regional_summary.round(8))

In [ ]:
display(curve_summary.round(8))

## Experiment 3 — Iteration-wise propagation

Variance after every stage x0 … x8, not just the final output.

*Expected: 1.00 → 1.737 → 3.014 → 5.207 → 8.991 → 15.419 → 26.409 → 45.246 → 78.349
at sigma = 0.005.*

In [ ]:

# ============================================================
# EXPERIMENT 3 — ITERATION-WISE NOISE PROPAGATION
#
# 2A: Frozen clean-image curves
# 2B: End-to-end noisy-image curve prediction
#
# Measures temporal variance after every curve iteration.
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

EXP3_DIR = Path(
    "/content/drive/MyDrive/zero-dce/results/phase_a/experiment_3"
)

EXP3_DIR.mkdir(parents=True, exist_ok=True)

EXP3_SIGMAS = SIGMAS
EXP3_TRIALS = N_TRIALS
EXP3_BASE_SEED = BASE_SEED

NUM_ITERATIONS = 8

image_paths = sorted([
    p for p in LOW_DIR.iterdir()
    if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]
])

print("Images:", len(image_paths))
print("Noise levels:", EXP3_SIGMAS)
print("Trials per noise level:", EXP3_TRIALS)
print("Iterations:", NUM_ITERATIONS)
print("Device:", device)


# ------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------

def exp3_sample_variance(sum_x, sum_x2, n):
    """
    Unbiased sample variance at every pixel/channel,
    calculated across independent noise realizations.
    """
    variance = (
        sum_x2 - (sum_x * sum_x) / n
    ) / (n - 1)

    return torch.clamp(variance, min=0.0)


def exp3_mean(tensor):
    return float(tensor.mean().item())


def exp3_regional_variance(variance_map, clean):
    """
    Average temporal variance across RGB channels,
    then aggregate pixels using original clean-image
    luminance masks.
    """

    luminance = (
        0.299 * clean[:, 0:1]
        + 0.587 * clean[:, 1:2]
        + 0.114 * clean[:, 2:3]
    )

    variance_luminance = variance_map.mean(
        dim=1, keepdim=True
    )

    output = {}

    for region, (lower, upper) in BRIGHTNESS_BINS.items():

        mask = (
            (luminance >= lower)
            & (luminance < upper)
        )

        pixel_count = int(mask.sum().item())

        if pixel_count > 0:
            output[region] = {
                "variance": float(
                    variance_luminance[mask].mean().item()
                ),
                "pixel_count": pixel_count
            }
        else:
            output[region] = {
                "variance": np.nan,
                "pixel_count": 0
            }

    return output


def exp3_get_curve_steps(curves, height, width):
    """
    Convert [B, 24, H, W] into [B, 8, 3, H, W].
    """
    return curves.reshape(
        curves.shape[0],
        8,
        3,
        height,
        width
    )


def exp3_clean_trajectory(clean, curves):
    """
    Calculate clean trajectory:
    x0, x1, ..., x8

    Also calculate cumulative local Jacobian:
    J0, J1, ..., J8
    """

    curve_steps = exp3_get_curve_steps(
        curves,
        clean.shape[-2],
        clean.shape[-1]
    )

    x = clean.clone()
    J = torch.ones_like(clean)

    clean_stages = [x.clone()]
    jacobian_stages = [J.clone()]

    for iteration in range(8):

        A = curve_steps[:, iteration]

        local_derivative = (
            1.0 + A * (2.0 * x - 1.0)
        )

        J = J * local_derivative

        x = x + A * (x * x - x)

        clean_stages.append(x.clone())
        jacobian_stages.append(J.clone())

    return clean_stages, jacobian_stages


def exp3_noisy_trajectory(noisy, curves):
    """
    Apply eight iterations to a noisy image.

    The curves supplied can be either:
    - Clean curves for frozen branch 2A.
    - Noisy-image curves for end-to-end branch 2B.

    Returns x0, x1, ..., x8.
    """

    curve_steps = exp3_get_curve_steps(
        curves,
        noisy.shape[-2],
        noisy.shape[-1]
    )

    x = noisy.clone()
    stages = [x.clone()]

    for iteration in range(8):

        A = curve_steps[:, iteration]

        x = x + A * (x * x - x)

        stages.append(x.clone())

    return stages


# ------------------------------------------------------------
# 3. RESULT STORAGE
# ------------------------------------------------------------

all_rows = []
regional_rows = []

metadata = {
    "experiment": "Experiment 3 - Iteration-wise Noise Propagation",
    "date": datetime.now().isoformat(),
    "noise_levels": EXP3_SIGMAS,
    "trials_per_image_sigma": EXP3_TRIALS,
    "iterations": NUM_ITERATIONS,
    "brightness_masks": "Fixed using original clean image luminance",
    "input_noise_clipping": False,
    "jacobian_branch": "Frozen curves only",
    "images": []
}


# ------------------------------------------------------------
# 4. PROCESS EACH IMAGE
# ------------------------------------------------------------

for image_index, image_path in enumerate(
    tqdm(image_paths, desc="Experiment 3")
):

    image_name = image_path.name

    clean = load_rgb_image(image_path)

    height, width = clean.shape[-2:]

    # --------------------------------------------------------
    # Predict clean-image curves
    # --------------------------------------------------------

    with torch.no_grad():

        _, _, clean_curves = model(clean)

        clean_stages_gpu, jacobian_stages_gpu = (
            exp3_clean_trajectory(clean, clean_curves)
        )

    # --------------------------------------------------------
    # Move reference trajectories to CPU
    # --------------------------------------------------------

    clean_cpu = clean.detach().cpu().double()

    clean_stages = [
        stage.detach().cpu().double()
        for stage in clean_stages_gpu
    ]

    jacobian_stages = [
        stage.detach().cpu().double()
        for stage in jacobian_stages_gpu
    ]

    clean_curve_steps = clean_curves.reshape(
        1, 8, 3, height, width
    )

    # Record region pixel counts once per image
    original_luminance = (
        0.299 * clean_cpu[:, 0:1]
        + 0.587 * clean_cpu[:, 1:2]
        + 0.114 * clean_cpu[:, 2:3]
    )

    region_pixel_counts = {}

    for region, (lower, upper) in BRIGHTNESS_BINS.items():

        mask = (
            (original_luminance >= lower)
            & (original_luminance < upper)
        )

        region_pixel_counts[region] = int(
            mask.sum().item()
        )

    metadata["images"].append({
        "image": image_name,
        "height": height,
        "width": width,
        "region_pixel_counts": region_pixel_counts
    })


    # --------------------------------------------------------
    # 5. PROCESS EACH NOISE LEVEL
    # --------------------------------------------------------

    for sigma_index, sigma in enumerate(EXP3_SIGMAS):

        # Nine stages:
        # Stage 0 = input
        # Stage 1 = after curve 1
        # ...
        # Stage 8 = after curve 8

        branch_names = [
            "frozen",
            "end_to_end"
        ]

        sums = {
            branch: [
                torch.zeros_like(clean_cpu)
                for _ in range(9)
            ]
            for branch in branch_names
        }

        sums_sq = {
            branch: [
                torch.zeros_like(clean_cpu)
                for _ in range(9)
            ]
            for branch in branch_names
        }

        # Input noise statistics
        input_sum = torch.zeros_like(clean_cpu)
        input_sum_sq = torch.zeros_like(clean_cpu)


        # ----------------------------------------------------
        # 6. INDEPENDENT NOISE REALIZATIONS
        # ----------------------------------------------------

        for trial in range(EXP3_TRIALS):

            seed = (
                EXP3_BASE_SEED
                + image_index * 10000000
                + sigma_index * 100000
                + trial
            )

            generator = torch.Generator(device=device)
            generator.manual_seed(seed)

            noise = torch.randn(
                clean.shape,
                generator=generator,
                device=device,
                dtype=clean.dtype
            ) * sigma

            noisy = clean + noise


            # ------------------------------------------------
            # 2A: FROZEN CURVES
            # ------------------------------------------------

            with torch.no_grad():

                frozen_stages_gpu = exp3_noisy_trajectory(
                    noisy,
                    clean_curves
                )


            # ------------------------------------------------
            # 2B: END-TO-END
            # Predict new curves from each noisy image
            # ------------------------------------------------

            with torch.no_grad():

                _, _, noisy_curves = model(noisy)

                end_to_end_stages_gpu = exp3_noisy_trajectory(
                    noisy,
                    noisy_curves
                )


            # ------------------------------------------------
            # MOVE STAGES TO CPU
            # ------------------------------------------------

            frozen_stages = [
                stage.detach().cpu().double()
                for stage in frozen_stages_gpu
            ]

            end_to_end_stages = [
                stage.detach().cpu().double()
                for stage in end_to_end_stages_gpu
            ]

            noise_cpu = noise.detach().cpu().double()


            # ------------------------------------------------
            # ACCUMULATE INPUT NOISE
            # ------------------------------------------------

            input_sum += noise_cpu
            input_sum_sq += noise_cpu.square()


            # ------------------------------------------------
            # ACCUMULATE RESIDUALS AT EVERY ITERATION
            #
            # Residual = noisy trajectory - clean trajectory
            #
            # The clean reference at the same iteration is
            # subtracted to isolate the perturbation response.
            # ------------------------------------------------

            for stage_index in range(9):

                frozen_residual = (
                    frozen_stages[stage_index]
                    - clean_stages[stage_index]
                )

                end_to_end_residual = (
                    end_to_end_stages[stage_index]
                    - clean_stages[stage_index]
                )

                sums["frozen"][stage_index] += (
                    frozen_residual
                )

                sums_sq["frozen"][stage_index] += (
                    frozen_residual.square()
                )

                sums["end_to_end"][stage_index] += (
                    end_to_end_residual
                )

                sums_sq["end_to_end"][stage_index] += (
                    end_to_end_residual.square()
                )


            # Release trial-specific tensors
            del noise, noisy
            del frozen_stages_gpu, end_to_end_stages_gpu
            del frozen_stages, end_to_end_stages
            del noisy_curves, noise_cpu


        # ----------------------------------------------------
        # 7. CALCULATE TEMPORAL VARIANCE AT EACH STAGE
        # ----------------------------------------------------

        input_variance_map = exp3_sample_variance(
            input_sum,
            input_sum_sq,
            EXP3_TRIALS
        )

        input_variance = exp3_mean(
            input_variance_map
        )

        # Store the per-stage results for each branch
        branch_variances = {
            branch: []
            for branch in branch_names
        }

        for branch in branch_names:

            for stage_index in range(9):

                variance_map = exp3_sample_variance(
                    sums[branch][stage_index],
                    sums_sq[branch][stage_index],
                    EXP3_TRIALS
                )

                branch_variances[branch].append(
                    variance_map
                )


        # ----------------------------------------------------
        # 8. ITERATION-WISE ANALYSIS
        # ----------------------------------------------------

        for stage_index in range(9):

            frozen_variance_map = (
                branch_variances["frozen"][stage_index]
            )

            end_to_end_variance_map = (
                branch_variances["end_to_end"][stage_index]
            )

            # Mean temporal variance
            frozen_variance = exp3_mean(
                frozen_variance_map
            )

            end_to_end_variance = exp3_mean(
                end_to_end_variance_map
            )

            # Cumulative Jacobian prediction
            J = jacobian_stages[stage_index]

            predicted_variance_map = (
                input_variance_map * J.square()
            )

            predicted_variance = exp3_mean(
                predicted_variance_map
            )

            # Amplification relative to original input noise
            frozen_amplification = (
                frozen_variance / input_variance
                if input_variance > 0 else np.nan
            )

            end_to_end_amplification = (
                end_to_end_variance / input_variance
                if input_variance > 0 else np.nan
            )

            predicted_amplification = (
                predicted_variance / input_variance
                if input_variance > 0 else np.nan
            )

            # Prediction error is meaningful for frozen curves
            prediction_error = (
                abs(frozen_variance - predicted_variance)
                / max(predicted_variance, 1e-15)
                * 100
            )

            row = {
                "image": image_name,
                "sigma": sigma,
                "stage": stage_index,
                "iteration": (
                    "input"
                    if stage_index == 0
                    else stage_index
                ),

                "input_variance": input_variance,

                "frozen_temporal_variance": frozen_variance,

                "end_to_end_temporal_variance": (
                    end_to_end_variance
                ),

                "frozen_variance_amplification": (
                    frozen_amplification
                ),

                "end_to_end_variance_amplification": (
                    end_to_end_amplification
                ),

                "jacobian_predicted_variance": (
                    predicted_variance
                ),

                "jacobian_predicted_amplification": (
                    predicted_amplification
                ),

                "frozen_jacobian_relative_error_percent": (
                    prediction_error
                )
            }

            all_rows.append(row)


            # ------------------------------------------------
            # REGIONAL VARIANCE
            # ------------------------------------------------

            frozen_regions = exp3_regional_variance(
                frozen_variance_map,
                clean_cpu
            )

            end_to_end_regions = exp3_regional_variance(
                end_to_end_variance_map,
                clean_cpu
            )

            input_regions = exp3_regional_variance(
                input_variance_map,
                clean_cpu
            )

            predicted_regions = exp3_regional_variance(
                predicted_variance_map,
                clean_cpu
            )

            for region in BRIGHTNESS_BINS:

                regional_rows.append({

                    "image": image_name,
                    "sigma": sigma,
                    "stage": stage_index,
                    "iteration": (
                        "input"
                        if stage_index == 0
                        else stage_index
                    ),

                    "region": region,

                    "pixel_count": (
                        region_pixel_counts[region]
                    ),

                    "input_variance": (
                        input_regions[region]["variance"]
                    ),

                    "frozen_variance": (
                        frozen_regions[region]["variance"]
                    ),

                    "end_to_end_variance": (
                        end_to_end_regions[region]["variance"]
                    ),

                    "jacobian_predicted_variance": (
                        predicted_regions[region]["variance"]
                    )
                })


        # ----------------------------------------------------
        # 9. PROGRESS
        # ----------------------------------------------------

        stage8 = all_rows[-1]

        print(
            f"{image_name} | sigma={sigma:.3f} | "
            f"Final 2A={stage8['frozen_variance_amplification']:.3f}x | "
            f"Final 2B={stage8['end_to_end_variance_amplification']:.3f}x | "
            f"Jacobian={stage8['jacobian_predicted_amplification']:.3f}x"
        )


        # Release sigma-specific accumulators
        del sums, sums_sq
        del branch_variances
        del input_sum, input_sum_sq
        del input_variance_map

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    # --------------------------------------------------------
    # 10. RELEASE IMAGE-SPECIFIC TENSORS
    # --------------------------------------------------------

    del clean
    del clean_curves
    del clean_stages_gpu
    del jacobian_stages_gpu
    del clean_stages
    del jacobian_stages
    del clean_curve_steps
    del clean_cpu

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\nExperiment 3 completed successfully.")


In [ ]:
# Convert to DataFrames
exp3_df = pd.DataFrame(all_rows)
exp3_regional_df = pd.DataFrame(regional_rows)

# Save detailed per-image results
exp3_df.to_csv(
    EXP3_DIR / "experiment_3_per_image.csv",
    index=False
)

exp3_regional_df.to_csv(
    EXP3_DIR / "experiment_3_regional_per_image.csv",
    index=False
)

# Aggregate across images, giving each image equal weight
exp3_summary = (
    exp3_df
    .groupby(["sigma", "stage"], as_index=False)
    .agg({
        "frozen_temporal_variance": "mean",
        "end_to_end_temporal_variance": "mean",
        "frozen_variance_amplification": "mean",
        "end_to_end_variance_amplification": "mean",
        "jacobian_predicted_variance": "mean",
        "jacobian_predicted_amplification": "mean",
        "frozen_jacobian_relative_error_percent": "mean"
    })
)

exp3_summary.to_csv(
    EXP3_DIR / "experiment_3_summary.csv",
    index=False
)

# Aggregate regional results
exp3_regional_summary = (
    exp3_regional_df
    .groupby(
        ["sigma", "stage", "region"],
        as_index=False
    )
    .agg({
        "pixel_count": "mean",
        "input_variance": "mean",
        "frozen_variance": "mean",
        "end_to_end_variance": "mean",
        "jacobian_predicted_variance": "mean"
    })
)

exp3_regional_summary.to_csv(
    EXP3_DIR / "experiment_3_regional_summary.csv",
    index=False
)

# Save metadata
metadata["number_of_images"] = len(image_paths)
metadata["number_of_trials"] = EXP3_TRIALS

with open(
    EXP3_DIR / "experiment_3_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print("Experiment 3 results saved to:", EXP3_DIR)

In [ ]:
display(
    exp3_summary.round(6)
)

In [ ]:
display(
    exp3_regional_summary.round(8)
)

In [ ]:
import matplotlib.pyplot as plt

for sigma in EXP3_SIGMAS:

    subset = exp3_summary[
        exp3_summary["sigma"] == sigma
    ].sort_values("stage")

    plt.figure(figsize=(10, 5))

    plt.plot(
        subset["stage"],
        subset["frozen_variance_amplification"],
        marker="o",
        label="Frozen curves (2A)"
    )

    plt.plot(
        subset["stage"],
        subset["end_to_end_variance_amplification"],
        marker="s",
        label="End-to-end (2B)"
    )

    plt.plot(
        subset["stage"],
        subset["jacobian_predicted_amplification"],
        marker="^",
        linestyle="--",
        label="First-order Jacobian"
    )

    plt.xlabel("Curve iteration")
    plt.ylabel("Variance amplification")
    plt.title(
        f"Iteration-wise Noise Propagation (sigma={sigma})"
    )

    plt.xticks(range(9), [
        "Input", "1", "2", "3", "4", "5", "6", "7", "8"
    ])

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Experiment 4 — Per-iteration, per-pixel sensitivity attribution

Local `g_n` and cumulative `J_n` along the clean trajectory, grouped by brightness.
Bootstrap CIs resample **images**, not pixels.

*Expected: mean local variance gain 1.7365 → 1.3753; mean cumulative 76.988 at
iteration 8; p99 cumulative 293.82.*

> Region sizes are severely imbalanced — `bright` is empty in 9 of 15 images and
> never exceeds 1,180 of 240,000 pixels. Treat mid-bright and bright conclusions
> as unsupported by this dataset.

In [ ]:

# ============================================================
# EXPERIMENT 4 — PER-ITERATION AND PER-PIXEL SENSITIVITY
#
# Goal:
# Attribute cumulative noise sensitivity to the eight
# Zero-DCE curve transformations.
#
# Uses:
#   - Original baseline checkpoint
#   - Clean-image predicted curves
#   - Clean-image trajectory
#
# No training or parameter updates.
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

EXP4_DIR = Path(
    "/content/drive/MyDrive/zero-dce/results/phase_a/experiment_4"
)
EXP4_DIR.mkdir(parents=True, exist_ok=True)

EXP4_BOOTSTRAP_SAMPLES = 10000
EXP4_BOOTSTRAP_SEED = 42026

EXP4_REGIONS = {
    "dark": (0.0, 0.10),
    "mid_dark": (0.10, 0.25),
    "mid_bright": (0.25, 0.50),
    "bright": (0.50, 1.01),
}

image_paths = sorted([
    p for p in LOW_DIR.iterdir()
    if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]
])

assert len(image_paths) > 0, "No images found."

model.eval()

print("Experiment 4")
print("Images:", len(image_paths))
print("Iterations:", 8)
print("Bootstrap samples:", EXP4_BOOTSTRAP_SAMPLES)
print("Device:", device)


# ------------------------------------------------------------
# 2. METRIC FUNCTION
# ------------------------------------------------------------

def exp4_metrics(gain_values, cumulative_values):
    """
    gain_values:
        Individual iteration derivative g_n.

    cumulative_values:
        Cumulative derivative J_n.

    Metrics are calculated across all selected pixels
    and RGB channels.
    """

    g = np.asarray(gain_values, dtype=np.float64).reshape(-1)
    J = np.asarray(cumulative_values, dtype=np.float64).reshape(-1)

    abs_g = np.abs(g)
    g_squared = g ** 2
    abs_J = np.abs(J)
    J_squared = J ** 2

    return {
        "sample_count": int(len(g)),

        # Signed derivative behaviour
        "mean_signed_gain": float(np.mean(g)),
        "median_signed_gain": float(np.median(g)),

        # Magnitude of local sensitivity
        "mean_absolute_gain": float(np.mean(abs_g)),
        "median_absolute_gain": float(np.median(abs_g)),

        "gain_abs_p05": float(np.percentile(abs_g, 5)),
        "gain_abs_p25": float(np.percentile(abs_g, 25)),
        "gain_abs_p75": float(np.percentile(abs_g, 75)),
        "gain_abs_p95": float(np.percentile(abs_g, 95)),

        # Variance response
        "mean_local_variance_gain": float(np.mean(g_squared)),
        "median_local_variance_gain": float(np.median(g_squared)),

        # Local amplification / attenuation
        "fraction_abs_gain_gt_1": float(np.mean(abs_g > 1)),
        "fraction_abs_gain_lt_1": float(np.mean(abs_g < 1)),
        "fraction_abs_gain_eq_1": float(np.mean(np.isclose(abs_g, 1))),

        # Signed expansion / contraction
        "fraction_gain_gt_1": float(np.mean(g > 1)),
        "fraction_gain_lt_minus_1": float(np.mean(g < -1)),
        "fraction_gain_between_minus1_and_1": float(
            np.mean((g > -1) & (g < 1))
        ),

        # Cumulative sensitivity
        "mean_absolute_cumulative_gain": float(np.mean(abs_J)),
        "mean_cumulative_variance_gain": float(np.mean(J_squared)),
        "median_cumulative_variance_gain": float(np.median(J_squared)),

        "cumulative_variance_gain_p95": float(
            np.percentile(J_squared, 95)
        ),
        "cumulative_variance_gain_p99": float(
            np.percentile(J_squared, 99)
        ),
    }


# ------------------------------------------------------------
# 3. STORAGE
# ------------------------------------------------------------

iteration_rows = []
regional_rows = []
cumulative_rows = []
image_metadata = []


# ------------------------------------------------------------
# 4. PROCESS EACH IMAGE
# ------------------------------------------------------------

for image_index, image_path in enumerate(
    tqdm(image_paths, desc="Experiment 4")
):

    image_name = image_path.name

    clean = load_rgb_image(image_path)

    height, width = clean.shape[-2:]

    # Predict curves from the clean image
    with torch.no_grad():
        _, _, clean_curves = model(clean)

    curve_steps = clean_curves.reshape(
        1, 8, 3, height, width
    )

    # Original clean-image luminance
    clean_luminance = (
        0.299 * clean[:, 0:1]
        + 0.587 * clean[:, 1:2]
        + 0.114 * clean[:, 2:3]
    )

    # Fixed brightness masks from original clean image
    region_masks = {}

    for region, (lower, upper) in EXP4_REGIONS.items():

        region_masks[region] = (
            (clean_luminance >= lower)
            & (clean_luminance < upper)
        )

    region_pixel_counts = {
        region: int(mask.sum().item())
        for region, mask in region_masks.items()
    }

    # Start clean trajectory
    x = clean.clone()

    # Cumulative derivative starts at 1
    J = torch.ones_like(clean)

    # Track cumulative log variance contribution.
    # Sum of log(g_n^2) equals log(J_n^2), except at zero.
    cumulative_log_variance_gain = torch.zeros_like(clean)

    # --------------------------------------------------------
    # ITERATE THROUGH THE EIGHT CURVE TRANSFORMATIONS
    # --------------------------------------------------------

    for iteration in range(8):

        A = curve_steps[:, iteration]

        # Local derivative evaluated at the current clean state
        g = 1.0 + A * (2.0 * x - 1.0)

        # Individual local variance gain
        local_variance_gain = g.square()

        # Update cumulative Jacobian
        J = J * g

        cumulative_variance_gain = J.square()

        # Per-iteration contribution to log variance gain
        # Clamp only for logarithm numerical stability.
        # Original g and g^2 are retained unchanged.
        log_contribution = torch.log(
            torch.clamp(local_variance_gain, min=1e-30)
        )

        cumulative_log_variance_gain += log_contribution

        # Move measurement maps to CPU
        g_cpu = g.detach().cpu().double()
        local_gain_cpu = (
            local_variance_gain.detach().cpu().double()
        )
        J_cpu = J.detach().cpu().double()
        cumulative_gain_cpu = (
            cumulative_variance_gain.detach().cpu().double()
        )
        log_contribution_cpu = (
            log_contribution.detach().cpu().double()
        )
        cumulative_log_cpu = (
            cumulative_log_variance_gain.detach().cpu().double()
        )

        clean_cpu = clean.detach().cpu().double()

        # ----------------------------------------------------
        # OVERALL PIXEL / CHANNEL DISTRIBUTION
        # ----------------------------------------------------

        overall = exp4_metrics(
            g_cpu.numpy(),
            J_cpu.numpy()
        )

        overall.update({
            "image": image_name,
            "iteration": iteration + 1,
            "region": "all"
        })

        iteration_rows.append(overall)

        # ----------------------------------------------------
        # BRIGHTNESS-SPECIFIC DISTRIBUTIONS
        # ----------------------------------------------------

        for region, mask_gpu in region_masks.items():

            mask_cpu = mask_gpu.detach().cpu().numpy().astype(bool)

            # Broadcast spatial mask over RGB channels
            mask_rgb = np.broadcast_to(
                mask_cpu,
                g_cpu.shape
            )

            g_region = g_cpu.numpy()[mask_rgb]
            J_region = J_cpu.numpy()[mask_rgb]

            if len(g_region) > 0:

                region_metrics = exp4_metrics(
                    g_region,
                    J_region
                )

            else:

                region_metrics = {
                    "sample_count": 0
                }

            region_metrics.update({
                "image": image_name,
                "iteration": iteration + 1,
                "region": region,
                "region_pixel_count": region_pixel_counts[region]
            })

            regional_rows.append(region_metrics)

            # ------------------------------------------------
            # CUMULATIVE SENSITIVITY BY REGION
            # ------------------------------------------------

            local_gain_region = (
                local_gain_cpu.numpy()[mask_rgb]
            )

            cumulative_gain_region = (
                cumulative_gain_cpu.numpy()[mask_rgb]
            )

            log_contribution_region = (
                log_contribution_cpu.numpy()[mask_rgb]
            )

            cumulative_log_region = (
                cumulative_log_cpu.numpy()[mask_rgb]
            )

            cumulative_rows.append({

                "image": image_name,
                "iteration": iteration + 1,
                "region": region,

                "region_pixel_count": region_pixel_counts[region],
                "channel_sample_count": int(len(g_region)),

                "mean_local_variance_gain": (
                    float(np.mean(local_gain_region))
                    if len(local_gain_region) else np.nan
                ),

                "mean_cumulative_variance_gain": (
                    float(np.mean(cumulative_gain_region))
                    if len(cumulative_gain_region) else np.nan
                ),

                "mean_log_local_variance_gain": (
                    float(np.mean(log_contribution_region))
                    if len(log_contribution_region) else np.nan
                ),

                "mean_cumulative_log_variance_gain": (
                    float(np.mean(cumulative_log_region))
                    if len(cumulative_log_region) else np.nan
                )
            })

        # ----------------------------------------------------
        # ADVANCE CLEAN TRAJECTORY
        # ----------------------------------------------------

        x = x + A * (x * x - x)

    # --------------------------------------------------------
    # IMAGE METADATA
    # --------------------------------------------------------

    image_metadata.append({
        "image": image_name,
        "height": height,
        "width": width,
        **{
            f"pixels_{region}": count
            for region, count in region_pixel_counts.items()
        }
    })

    del clean
    del clean_curves
    del curve_steps
    del x, J
    del cumulative_log_variance_gain

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\nExperiment 4 sensitivity extraction completed.")


# ------------------------------------------------------------
# 5. CREATE DATAFRAMES
# ------------------------------------------------------------

exp4_iteration_df = pd.DataFrame(iteration_rows)
exp4_regional_df = pd.DataFrame(regional_rows)
exp4_cumulative_df = pd.DataFrame(cumulative_rows)
exp4_metadata_df = pd.DataFrame(image_metadata)


# ------------------------------------------------------------
# 6. SAVE PER-IMAGE DATA
# ------------------------------------------------------------

exp4_iteration_df.to_csv(
    EXP4_DIR / "experiment_4_iteration_per_image.csv",
    index=False
)

exp4_regional_df.to_csv(
    EXP4_DIR / "experiment_4_regional_per_image.csv",
    index=False
)

exp4_cumulative_df.to_csv(
    EXP4_DIR / "experiment_4_cumulative_per_image.csv",
    index=False
)

exp4_metadata_df.to_csv(
    EXP4_DIR / "experiment_4_image_metadata.csv",
    index=False
)

print("Per-image measurements saved.")


In [ ]:

# ============================================================
# EXPERIMENT 4 — IMAGE-LEVEL STATISTICS AND BOOTSTRAP CIs
# ============================================================

BOOTSTRAP_RNG = np.random.default_rng(
    EXP4_BOOTSTRAP_SEED
)

def exp4_bootstrap_ci(values, n_boot=10000):
    """
    Percentile bootstrap confidence interval for the mean.
    Each value represents one image-level measurement.
    """

    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    n = len(values)

    if n == 0:
        return np.nan, np.nan, np.nan

    mean_value = float(np.mean(values))

    if n == 1:
        return mean_value, np.nan, np.nan

    indices = BOOTSTRAP_RNG.integers(
        0,
        n,
        size=(n_boot, n)
    )

    bootstrap_means = values[indices].mean(axis=1)

    lower = float(np.percentile(bootstrap_means, 2.5))
    upper = float(np.percentile(bootstrap_means, 97.5))

    return mean_value, lower, upper


def exp4_aggregate_with_ci(
    dataframe,
    group_columns,
    metric_columns,
    n_boot=10000
):
    """
    Aggregate image-level observations.
    The bootstrap resamples images, not pixels.
    """

    output_rows = []

    grouped = dataframe.groupby(
        group_columns,
        dropna=False
    )

    for group_key, group in grouped:

        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        base = dict(zip(group_columns, group_key))

        for metric in metric_columns:

            values = (
                group[metric]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
                .to_numpy(dtype=np.float64)
            )

            mean_value, ci_lower, ci_upper = (
                exp4_bootstrap_ci(
                    values,
                    n_boot=n_boot
                )
            )

            output_rows.append({
                **base,
                "metric": metric,
                "number_of_images": int(len(values)),
                "mean": mean_value,
                "median": (
                    float(np.median(values))
                    if len(values) else np.nan
                ),
                "std_across_images": (
                    float(np.std(values, ddof=1))
                    if len(values) > 1 else np.nan
                ),
                "ci95_lower": ci_lower,
                "ci95_upper": ci_upper
            })

    return pd.DataFrame(output_rows)


# ------------------------------------------------------------
# OVERALL ITERATION STATISTICS
# ------------------------------------------------------------

overall_metrics = [
    "mean_signed_gain",
    "mean_absolute_gain",
    "median_absolute_gain",
    "mean_local_variance_gain",
    "fraction_abs_gain_gt_1",
    "fraction_abs_gain_lt_1",
    "fraction_gain_gt_1",
    "fraction_gain_lt_minus_1",
    "mean_cumulative_variance_gain",
    "median_cumulative_variance_gain",
    "cumulative_variance_gain_p95",
    "cumulative_variance_gain_p99"
]

exp4_overall_ci = exp4_aggregate_with_ci(
    exp4_iteration_df,
    ["iteration"],
    overall_metrics,
    n_boot=EXP4_BOOTSTRAP_SAMPLES
)


# ------------------------------------------------------------
# REGIONAL ITERATION STATISTICS
# ------------------------------------------------------------

regional_metrics = [
    "mean_signed_gain",
    "mean_absolute_gain",
    "mean_local_variance_gain",
    "fraction_abs_gain_gt_1",
    "fraction_abs_gain_lt_1",
    "fraction_gain_gt_1",
    "fraction_gain_lt_minus_1",
    "mean_cumulative_variance_gain",
    "cumulative_variance_gain_p95",
    "cumulative_variance_gain_p99"
]

exp4_regional_ci = exp4_aggregate_with_ci(
    exp4_regional_df,
    ["region", "iteration"],
    regional_metrics,
    n_boot=EXP4_BOOTSTRAP_SAMPLES
)


# ------------------------------------------------------------
# CUMULATIVE LOG CONTRIBUTION STATISTICS
# ------------------------------------------------------------

cumulative_metrics = [
    "mean_local_variance_gain",
    "mean_cumulative_variance_gain",
    "mean_log_local_variance_gain",
    "mean_cumulative_log_variance_gain"
]

exp4_cumulative_ci = exp4_aggregate_with_ci(
    exp4_cumulative_df,
    ["region", "iteration"],
    cumulative_metrics,
    n_boot=EXP4_BOOTSTRAP_SAMPLES
)


# ------------------------------------------------------------
# SAVE AGGREGATED RESULTS
# ------------------------------------------------------------

exp4_overall_ci.to_csv(
    EXP4_DIR / "experiment_4_overall_bootstrap_ci.csv",
    index=False
)

exp4_regional_ci.to_csv(
    EXP4_DIR / "experiment_4_regional_bootstrap_ci.csv",
    index=False
)

exp4_cumulative_ci.to_csv(
    EXP4_DIR / "experiment_4_cumulative_bootstrap_ci.csv",
    index=False
)


# ------------------------------------------------------------
# SAVE EXPERIMENT METADATA
# ------------------------------------------------------------

exp4_metadata = {
    "experiment": "Experiment 4 - Sensitivity Attribution",
    "date": datetime.now().isoformat(),
    "images": len(image_paths),
    "iterations": 8,
    "bootstrap_samples": EXP4_BOOTSTRAP_SAMPLES,
    "bootstrap_unit": "Image",
    "confidence_interval": "95% percentile bootstrap",
    "brightness_regions": EXP4_REGIONS,
    "derivative": "g_n = 1 + A_n * (2*x_(n-1) - 1)",
    "cumulative_jacobian": "J_n = product(g_i, i=1..n)",
    "variance_gain": "g_n^2",
    "cumulative_variance_gain": "J_n^2",
    "model_training": False
}

with open(
    EXP4_DIR / "experiment_4_metadata.json",
    "w"
) as f:
    json.dump(exp4_metadata, f, indent=4)


print("Statistical aggregation completed.")
print("Results saved to:", EXP4_DIR)


In [ ]:
# ============================================================
# DISPLAY EXPERIMENT 4 RESULTS
# ============================================================

# 1. Overall iteration sensitivity
overall_display = exp4_overall_ci[
    exp4_overall_ci["metric"].isin([
        "mean_local_variance_gain",
        "fraction_abs_gain_gt_1",
        "mean_cumulative_variance_gain",
        "cumulative_variance_gain_p95",
        "cumulative_variance_gain_p99"
    ])
]

print("TABLE 1 — OVERALL ITERATION SENSITIVITY")

display(
    overall_display.sort_values(
        ["metric", "iteration"]
    ).round(5)
)


# 2. Regional sensitivity
regional_display = exp4_regional_ci[
    exp4_regional_ci["metric"].isin([
        "mean_local_variance_gain",
        "fraction_abs_gain_gt_1",
        "mean_cumulative_variance_gain",
        "cumulative_variance_gain_p95"
    ])
]

print("TABLE 2 — REGIONAL SENSITIVITY")

display(
    regional_display.sort_values(
        ["region", "metric", "iteration"]
    ).round(5)
)


# 3. Log contribution
print("TABLE 3 — LOG VARIANCE CONTRIBUTION")

display(
    exp4_cumulative_ci[
        exp4_cumulative_ci["metric"].isin([
            "mean_log_local_variance_gain",
            "mean_cumulative_log_variance_gain"
        ])
    ].sort_values(
        ["region", "metric", "iteration"]
    ).round(5)
)


# 4. Actual sample sizes
print("TABLE 4 — REGION PIXEL COUNTS")

display(
    exp4_metadata_df
)

In [ ]:
# ============================================================
# EXPERIMENT 4 — VISUAL ANALYSIS
# ============================================================

import matplotlib.pyplot as plt


def exp4_plot_ci(
    dataframe,
    metric,
    title,
    ylabel,
    regions=None
):

    plt.figure(figsize=(10, 6))

    if regions is None:
        regions = ["all"]

    for region in regions:

        if "region" in dataframe.columns:
            subset = dataframe[
                dataframe["region"] == region
            ]
        else:
            subset = dataframe

        subset = subset[
            subset["metric"] == metric
        ].sort_values("iteration")

        if len(subset) == 0:
            continue

        x = subset["iteration"].to_numpy()
        y = subset["mean"].to_numpy()
        lower = subset["ci95_lower"].to_numpy()
        upper = subset["ci95_upper"].to_numpy()

        plt.plot(
            x,
            y,
            marker="o",
            label=region
        )

        if np.isfinite(lower).all() and np.isfinite(upper).all():
            plt.fill_between(
                x,
                lower,
                upper,
                alpha=0.15
            )

    plt.xlabel("Curve iteration")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(range(1, 9))
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# PLOT 1: INDIVIDUAL LOCAL VARIANCE GAIN
# ------------------------------------------------------------

exp4_plot_ci(
    exp4_regional_ci,
    metric="mean_local_variance_gain",
    title="Individual Iteration: Local Variance Gain",
    ylabel="Mean g_n squared",
    regions=[
        "dark",
        "mid_dark",
        "mid_bright",
        "bright"
    ]
)


# ------------------------------------------------------------
# PLOT 2: FRACTION OF PIXELS WITH |g| > 1
# ------------------------------------------------------------

exp4_plot_ci(
    exp4_regional_ci,
    metric="fraction_abs_gain_gt_1",
    title="Fraction of Pixels with Local Variance Amplification",
    ylabel="Fraction of pixels and channels",
    regions=[
        "dark",
        "mid_dark",
        "mid_bright",
        "bright"
    ]
)


# ------------------------------------------------------------
# PLOT 3: CUMULATIVE VARIANCE GAIN
# ------------------------------------------------------------

exp4_plot_ci(
    exp4_regional_ci,
    metric="mean_cumulative_variance_gain",
    title="Cumulative Variance Gain by Brightness Region",
    ylabel="Mean cumulative J_n squared",
    regions=[
        "dark",
        "mid_dark",
        "mid_bright",
        "bright"
    ]
)


# ------------------------------------------------------------
# PLOT 4: OVERALL CUMULATIVE VARIANCE GAIN
# ------------------------------------------------------------

exp4_plot_ci(
    exp4_overall_ci,
    metric="mean_cumulative_variance_gain",
    title="Overall Cumulative Variance Gain",
    ylabel="Mean cumulative J_n squared"
)


# ------------------------------------------------------------
# PLOT 5: LOG CONTRIBUTION OF EACH ITERATION
# ------------------------------------------------------------

exp4_plot_ci(
    exp4_cumulative_ci,
    metric="mean_log_local_variance_gain",
    title="Per-Iteration Log Variance Contribution",
    ylabel="Mean log(g_n squared)",
    regions=[
        "dark",
        "mid_dark",
        "mid_bright",
        "bright"
    ]
)

## Final validation — Experiment 3 vs Experiment 4

Joins measured frozen-curve response to the clean-trajectory Jacobian, keeping
**all** sigmas (an earlier version silently kept only the first via `groupby.first()`).

*Expected mean relative difference: 1.59% / 6.60% / 31.46%.*

In [ ]:

# ============================================================
# FINAL VALIDATION: EXPERIMENT 3 vs EXPERIMENT 4
# Retains every sigma instead of selecting the first.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

exp3_dir = Path(
    "/content/drive/MyDrive/zero-dce/results/phase_a/experiment_3"
)

# Locate the actual per-image CSV
candidate_files = list(exp3_dir.glob("*per_image*.csv"))

if not candidate_files:
    raise FileNotFoundError(
        "Experiment 3 per-image CSV not found. "
        "Check the Experiment 3 output directory."
    )

exp3_df = pd.read_csv(candidate_files[0])

required_exp3 = {
    "image",
    "sigma",
    "stage",
    "frozen_variance_amplification"
}

missing = required_exp3 - set(exp3_df.columns)

if missing:
    raise ValueError(
        f"Missing columns in Experiment 3 CSV: {missing}"
    )

# Keep final stage for every image and every sigma
exp3_final = exp3_df[
    exp3_df["stage"] == 8
][[
    "image",
    "sigma",
    "frozen_variance_amplification"
]].copy()

# Experiment 4: final cumulative Jacobian per image
exp4_final = exp4_iteration_df[
    exp4_iteration_df["iteration"] == 8
][[
    "image",
    "mean_cumulative_variance_gain"
]].copy()

exp4_final = exp4_final.rename(columns={
    "mean_cumulative_variance_gain":
        "exp4_jacobian_amplification"
})

# Merge: every image and every sigma
validation = exp3_final.merge(
    exp4_final,
    on="image",
    how="inner"
)

validation["relative_difference_percent"] = (
    np.abs(
        validation["frozen_variance_amplification"]
        - validation["exp4_jacobian_amplification"]
    )
    / validation["exp4_jacobian_amplification"].clip(lower=1e-12)
    * 100
)

# Aggregate separately for each sigma
validation_summary = (
    validation.groupby("sigma")
    .agg(
        number_of_images=("image", "nunique"),
        measured_mean=("frozen_variance_amplification", "mean"),
        jacobian_mean=("exp4_jacobian_amplification", "mean"),
        mean_relative_difference=(
            "relative_difference_percent", "mean"
        ),
        median_relative_difference=(
            "relative_difference_percent", "median"
        ),
        std_relative_difference=(
            "relative_difference_percent", "std"
        )
    )
    .reset_index()
)

validation.to_csv(
    EXP4_DIR / "experiment_3_vs_4_all_sigmas_per_image.csv",
    index=False
)

validation_summary.to_csv(
    EXP4_DIR / "experiment_3_vs_4_all_sigmas_summary.csv",
    index=False
)

print("FINAL VALIDATION — ALL NOISE LEVELS")
display(validation_summary.round(5))


---
## Phase A conclusions

- The eight-step recurrence reconstructs exactly.
- Curve response is strongly brightness dependent: dark amplifies, bright attenuates.
- Variance grows progressively through **all** eight iterations.
- Frozen and end-to-end responses are close; the ~1.5% end-to-end reduction is
  **not** evidence of denoising.
- The first-order Jacobian explains small perturbations well (1.59% at sigma=0.005)
  and degrades as perturbations grow (31.46% at sigma=0.020).

### Limits
Synthetic additive Gaussian noise on images that are already noisy; 15 images;
mid-bright and bright regions effectively absent; the frozen Jacobian excludes
the network's curve-prediction pathway.

### Important caveat for interpretation
`J` is simultaneously the noise gain **and** the local contrast gain. A variance
amplification of 76.99x is a std amplification of 8.77x, which is also the factor
by which dark-region contrast is amplified. Do not report the amplification figure
without this, and do not treat it on its own as evidence of quality degradation.